# CAM-LDS classification with local GGUF models

Run the original CAM-LDS queries through five local GGUF models, smallest first. Each model automatically appends each successful response to one CSV in the shared Google Drive output folder. No manual export is needed. Clear text sections and commented functions explain the flow: **construct a query → load a model → query the model → save a response**.

**Stable prompt IDs:** The first CSV column, `prompt id`, is the query's original one-based position in `llm_queries.json` (1–990 for the pinned dataset). Only valid responses are saved. If query 2 fails, the saved IDs might be `1, 3, 4`; they are never renumbered to close the gap.

Ordinary query errors do not stop the dataset. A caught loading/engine failure moves to the next model. CSV write errors stop execution so saving problems are not hidden. One percentage bar per model shows attempted queries, saved responses, failures, and status.

The authors' queries, cleanup rules, and response schema are preserved. Local models, quantisations, context limits, output cap, and retry policy differ from the exact hosted-model experiment. Output is now a nine-column CSV, not the authors' response JSONL.

**Experiment outputs:** CSV only. No JSON, checkpoint, debugging, or backup files are created. Weights and installation caches stay on Colab's local disk.

## Before you begin

1. Upload this `.ipynb` through **File → Upload notebook** in Google Colab.
2. Select a **GPU** under **Runtime → Change runtime type**. The setup is intended for the T4 runtime discussed in our conversation.
3. Start with a fresh runtime and run Sections 1–10 in order. Section 11 starts inference.
4. For a new experiment, leave `RESUME_RUN_ID = None`. For an existing experiment, enter its filename timestamp in Section 3 before running setup.

All outputs go directly to `/content/drive/MyDrive/CAM_LDS_csv`. For example:

`llm_responses_qwen7b_camlds_20261009_063000_123456.csv`

The timestamp is generated automatically in **UTC**. All models in the same run share it. Model names and timestamps appear in filenames, not additional CSV columns.

## 1. Install the GPU inference engine

**Purpose:** Install a ready-made CUDA wheel instead of compiling `llama-cpp-python` from source.

Run this cell once in a fresh GPU runtime. It installs the verified CUDA 12.4 wheel for version 0.3.36, plus the CUDA 12 runtime and cuBLAS libraries it needs. Those libraries address the earlier `libcudart.so.12` import error on your CUDA 13 Colab machine.

The installation commands fail immediately if an installation fails; they do not silently fall back to a CPU build. If you previously imported another `llama_cpp` build in this runtime, restart the session before continuing.

In [2]:
import shutil
import subprocess
import sys

# Check that Colab actually assigned a GPU before downloading packages.
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Select a GPU runtime in Colab, then rerun this cell.")

subprocess.run(["nvidia-smi"], check=True)

# This is the official prebuilt CUDA 12.4 wheel, not a source archive.
CUDA_WHEEL_URL = (
    "https://github.com/abetlen/llama-cpp-python/releases/download/"
    "v0.3.36-cu124/"
    "llama_cpp_python-0.3.36-py3-none-manylinux_2_35_x86_64.whl"
)

# Replace any previous inference build while leaving other packages alone.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "--only-binary=:all:", "--force-reinstall", "--no-deps",
    CUDA_WHEEL_URL,
])

# Install the wheel's dependencies and the notebook utilities.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "--only-binary=:all:",
    "typing_extensions>=4.5.0", "numpy>=1.20.0", "diskcache>=5.6.1",
    "jinja2>=2.11.3", "huggingface_hub", "requests", "tqdm",
    "nvidia-cuda-runtime-cu12==12.4.127",
    "nvidia-cublas-cu12==12.4.5.8",
])

print("Installation complete. Continue to Section 2.")

Installation complete. Continue to Section 2.


## 2. Load the CUDA libraries and check the GPU build

**Purpose:** Make the installed CUDA 12 libraries available before importing the inference engine.

The runtime library, cuBLAS helper library, and cuBLAS library are loaded in that order. Keeping their handles in memory prevents them being released while inference is running.

The GPU-support check confirms that the installed engine can offload work. Actual layer placement is shown later when each model loads.

In [3]:
import ctypes
from importlib.metadata import distribution, version

# Retain these handles for the lifetime of this runtime.
CUDA_LIBRARY_HANDLES = []

cuda_libraries = [
    ("nvidia-cuda-runtime-cu12", "libcudart.so.12"),
    ("nvidia-cublas-cu12", "libcublasLt.so.12"),
    ("nvidia-cublas-cu12", "libcublas.so.12"),
]

for package_name, library_name in cuda_libraries:
    installed_package = distribution(package_name)
    matching_file = next(
        (item for item in installed_package.files or []
         if item.name == library_name),
        None,
    )
    if matching_file is None:
        raise RuntimeError(f"Cannot locate {library_name}; rerun Section 1.")

    library_path = installed_package.locate_file(matching_file)
    library_handle = ctypes.CDLL(str(library_path), mode=ctypes.RTLD_GLOBAL)
    CUDA_LIBRARY_HANDLES.append(library_handle)

# Import only after the shared libraries have been loaded.
import llama_cpp
from llama_cpp import Llama, LlamaGrammar

if version("llama-cpp-python") != "0.3.36":
    raise RuntimeError("This notebook expects llama-cpp-python 0.3.36.")
if not llama_cpp.llama_supports_gpu_offload():
    raise RuntimeError("The installed inference engine does not support GPU offloading.")

print(f"llama-cpp-python: {version('llama-cpp-python')}")
print("GPU offloading is supported.")

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 97250 MiB):
  Device 0: NVIDIA RTX PRO 6000 Blackwell Server Edition, compute capability 12.0, VMM: yes, VRAM: 97250 MiB


llama-cpp-python: 0.3.36
GPU offloading is supported.


## 3. Configure the experiment and connect Google Drive

**Purpose:** Keep the settings that visitors are most likely to change together in one cell.

- `SELECTED_MODELS` controls which models run. Its default order is smallest to largest.
- `CHECKPOINT_EVERY = 1` appends and flushes each successful response as soon as its query finishes. No manual CSV import or export is needed.
- `RESUME_RUN_ID = None` creates a new timestamp when this cell runs. Enter an existing timestamp to resume that experiment.
- `MAX_OUTPUT_TOKENS = 2048` is a generation cap, not a fixed response length. Shorter responses finish sooner.
- `CONTEXT_OVERRIDES` now sets 16,384 tokens for each selected model. The registry retains the original sizes. Context includes both the formatted prompt and generated output.

**Context limits:** The 16,384-token context and 2,048-token output allowance incorporate the larger limits suggested after query 2 stopped. These are starting values, not a guarantee every prompt or response fits. An oversized/truncated query is skipped and the next query runs. The notebook never shortens prompts. To restore the original contexts, use `CONTEXT_OVERRIDES = {}` and an output allowance of 1,024. Increasing context uses more memory; it does not change query cleaning.

**Resume settings:** The nine-column CSV cannot store model revisions, generation settings, or dependency versions. Keep those in this notebook, including pinned model revisions, and reuse them on resume. Rerunning Section 11 alone keeps the same run ID; rerunning this settings cell with `None` starts a new run.

The long-running cell uses percentage bars rather than per-query messages. Setup cells retain short checks. If you need to preserve revision pins before a restart, inspect the in-memory `MODEL_REVISIONS` dictionary and copy it into the settings cell; revisions are not printed during model execution.

**Resume the same output CSV:** Set `RESUME_RUN_ID` to the nine-column run's timestamp. Resume uses the highest saved prompt ID, not the CSV's row count. With successful responses only and no separate checkpoint file, failures after the last saved ID are not durably recorded and may be retried after restart. IDs before the highest saved ID are skipped, including gaps.
Use a fresh run ID for this nine-column format if your existing CSV has only eight columns. This notebook does not import other CSVs or overwrite incompatible files.


In [4]:
import copy
import csv
import gc
import io
import json
import os
import re

from datetime import datetime, timezone
from pathlib import Path

import requests
from google.colab import drive
from huggingface_hub import HfApi, hf_hub_download
from tqdm.auto import tqdm

# Keep only experiment CSVs in this shared Drive folder.
drive.mount("/content/drive")
OUTPUT_FOLDER = Path("/content/drive/MyDrive/CAM_LDS_csv")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# Weights and download caches stay on the faster local disk.
MODEL_DIR = Path("/content/models")

# Ascending model size. Remove entries here to run a subset.
SELECTED_MODELS = ["qwen1b", "hermes", "llama3b", "mistral7b", "qwen7b"]

# Use the timestamp already in your CSV filenames when resuming.
RESUME_RUN_ID = None
# RESUME_RUN_ID = "20261009_063000_123456"

RUN_ID = RESUME_RUN_ID or datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")

# Save each successful response immediately, without a manual export cell.
CHECKPOINT_EVERY = 1
TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 2048
SEED = 42

# These sizes batch tokens inside a prompt, not separate dataset queries.
GPU_BATCH_SIZE = 2048
GPU_MICROBATCH_SIZE = 512
GPU_FLASH_ATTENTION = True

# Give full log prompts more headroom, without changing their text.
CONTEXT_OVERRIDES = {model_key: 16384 for model_key in SELECTED_MODELS}
# To restore the original contexts, use {} and MAX_OUTPUT_TOKENS = 1024.
# These values are a starting point; some full prompts may still exceed them.

# Pin model commits here before a future runtime restart.
# Inspect MODEL_REVISIONS after downloading and save its dictionary here.
MODEL_REVISIONS = {}

# The stable source row number comes first; the other fields are unchanged.
CSV_COLUMNS = [
    "prompt id",
    "scenario", "variant", "step", "prompt", "ground truth",
    "techniques", "confidence", "explanation",
]

# Long quoted prompts and explanations must fit inside individual CSV fields.
csv.field_size_limit(min(sys.maxsize, 2**31 - 1))

if not isinstance(CHECKPOINT_EVERY, int) or CHECKPOINT_EVERY <= 0:
    raise ValueError("CHECKPOINT_EVERY must be a positive integer.")
if not isinstance(MAX_OUTPUT_TOKENS, int) or MAX_OUTPUT_TOKENS <= 0:
    raise ValueError("MAX_OUTPUT_TOKENS must be a positive integer.")
if not re.fullmatch(r"\d{8}_\d{6}(?:_\d{6})?", RUN_ID):
    raise ValueError("Use the timestamp from an existing CSV filename to resume.")



Mounted at /content/drive


## 3.1. Increase prompt-processing batches\
Detect the GPU assigned to this runtime and select starting batch sizes for long prompts. These settings control prompt-token processing, not concurrent dataset queries. Benchmark the selected sizes before starting the full experiment.

In [14]:
import subprocess

# Read the GPU currently allocated, rather than old saved cell outputs.
gpu_details = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader,nounits",
    ],
    text=True,
).strip().splitlines()[0]

gpu_name, gpu_memory = gpu_details.rsplit(",", 1)
gpu_name = gpu_name.strip()
gpu_memory_gib = float(gpu_memory.strip()) / 1024

# Starting sizes to test on the allocated GPU.
if "H100" in gpu_name and gpu_memory_gib >= 60:
    GPU_BATCH_SIZE, GPU_MICROBATCH_SIZE = 8192, 2048
elif "A100" in gpu_name and gpu_memory_gib >= 35:
    GPU_BATCH_SIZE, GPU_MICROBATCH_SIZE = 4096, 1024
elif "RTX PRO 6000" in gpu_name.upper() or "G4" in gpu_name.upper():
    GPU_BATCH_SIZE, GPU_MICROBATCH_SIZE = 4096, 1024
else:
    GPU_BATCH_SIZE, GPU_MICROBATCH_SIZE = 2048, 512

GPU_FLASH_ATTENTION = True

# Respect the CPU resources available to this runtime.
try:
    available_cpu_count = len(os.sched_getaffinity(0))
except AttributeError:
    available_cpu_count = os.cpu_count() or 1

CPU_THREADS = min(4, available_cpu_count)
CPU_BATCH_THREADS = min(8, available_cpu_count)

# Display one setup summary, without continuous printing.
{
    "GPU": gpu_name,
    "GPU memory GiB": round(gpu_memory_gib, 1),
    "prompt batch": GPU_BATCH_SIZE,
    "microbatch": GPU_MICROBATCH_SIZE,
}

{'GPU': 'NVIDIA RTX PRO 6000 Blackwell Server Edition',
 'GPU memory GiB': 95.6,
 'prompt batch': 4096,
 'microbatch': 1024}

In [6]:
from google.colab import userdata
from huggingface_hub import login

# Authenticate before any model downloads or benchmarking.
login(
    token=userdata.get("HF_TOKEN"),
    add_to_git_credential=False,
)

## 4. Define the models, smallest first

**Purpose:** Retain the original notebook's repositories, GGUF filenames, quantisations, and registry contexts. Section 3 overrides active contexts to 16,384 tokens.

| Order | Model key | Model | Quantisation | Original registry context |
| --- | --- | --- | --- | --- |
| 1 | `qwen1b` | Qwen2.5 1.5B Instruct | IQ1_M | 2,048 tokens |
| 2 | `hermes` | OpenHermes Qwen1.5 1.8B | IQ1_M | 2,048 tokens |
| 3 | `llama3b` | Llama 3.2 3B Instruct | IQ3_M | 4,096 tokens |
| 4 | `mistral7b` | Mistral 7B Instruct v0.3 | IQ1_M | 4,096 tokens |
| 5 | `qwen7b` | Qwen2.5 7B Instruct | IQ1_M | 4,096 tokens |

The two 7B models share the same parameter scale; Mistral precedes Qwen based on the approximate file sizes in the original notebook. This cell defines models without downloading them.

In [7]:
MODELS = {
    "qwen1b": {
        "name": "Qwen2.5 1.5B Instruct IQ1_M",
        "repo": "ThomasBaruzier/Qwen2.5-1.5B-Instruct-GGUF",
        "file": "Qwen2.5-1.5B-Instruct-IQ1_M.gguf",
        "ctx": 2048
    },
    "hermes": {
        "name": "OpenHermes Qwen1.5 1.8B IQ1_M",
        "repo": "mradermacher/OpenHermes-Qwen1.5-1.8B-i1-GGUF",
        "file": "OpenHermes-Qwen1.5-1.8B.i1-IQ1_M.gguf",
        "ctx": 2048
    },
    "llama3b": {
        "name": "Llama 3.2 3B Instruct IQ3_M",
        "repo": "bartowski/Llama-3.2-3B-Instruct-GGUF",
        "file": "Llama-3.2-3B-Instruct-IQ3_M.gguf",
        "ctx": 4096
    },
    "mistral7b": {
        "name": "Mistral 7B Instruct v0.3 IQ1_M",
        "repo": "bartowski/Mistral-7B-Instruct-v0.3-GGUF",
        "file": "Mistral-7B-Instruct-v0.3-IQ1_M.gguf",
        "ctx": 4096
    },
    "qwen7b": {
        "name": "Qwen2.5 7B Instruct IQ1_M",
        "repo": "ThomasBaruzier/Qwen2.5-7B-Instruct-GGUF",
        "file": "Qwen2.5-7B-Instruct-IQ1_M.gguf",
        "ctx": 4096
    }
}

# Catch a misspelled or duplicated model key before starting any downloads.
if not SELECTED_MODELS or len(set(SELECTED_MODELS)) != len(SELECTED_MODELS):
    raise ValueError("Choose at least one model and avoid duplicate keys.")
unknown_model_keys = set(SELECTED_MODELS) - set(MODELS)
if unknown_model_keys:
    raise ValueError(f"Unknown model keys: {unknown_model_keys}")
unknown_context_keys = set(CONTEXT_OVERRIDES) - set(MODELS)
if unknown_context_keys:
    raise ValueError(f"Unknown context override keys: {unknown_context_keys}")

for model_key in SELECTED_MODELS:
    context_size = CONTEXT_OVERRIDES.get(model_key, MODELS[model_key]["ctx"])
    if not isinstance(context_size, int) or context_size <= MAX_OUTPUT_TOKENS:
        raise ValueError(f"{model_key}: context must exceed the output-token cap.")
    print(f"{model_key}: {MODELS[model_key]['name']} | context = {context_size}")

qwen1b: Qwen2.5 1.5B Instruct IQ1_M | context = 16384
hermes: OpenHermes Qwen1.5 1.8B IQ1_M | context = 16384
llama3b: Llama 3.2 3B Instruct IQ3_M | context = 16384
mistral7b: Mistral 7B Instruct v0.3 IQ1_M | context = 16384
qwen7b: Qwen2.5 7B Instruct IQ1_M | context = 16384


## 5. Load the original CAM-LDS queries

**Purpose:** Read the original query file into memory, with no output copy of that file.

The authors' `llm_queries.json` contains one JSON object per line. Each object contains `scenario`, `variant`, `step`, `prompt`, and `ground_truth`.

This cell loads every row in its original order. It does not sample, sort, deduplicate, regenerate, compress, or edit the source records. The Git commit is pinned to the snapshot verified for this notebook. To resume an experiment started with another snapshot, use that experiment's original revision instead.

In [8]:
CAM_LDS_REVISION = "44028d8bd40a4a1d8bbbc6ee33261d47cb433827"
CAM_LDS_SOURCE_ROOT = (
    "https://raw.githubusercontent.com/"
    "ait-aecid/attack-manifestations-interpretation/"
    f"{CAM_LDS_REVISION}/"
)

query_download = requests.get(CAM_LDS_SOURCE_ROOT + "llm_queries.json", timeout=120)
query_download.raise_for_status()

# The .json extension is used for JSON Lines in the authors' repository.
QUERY_RECORDS = [
    json.loads(line)
    for line in query_download.content.decode("utf-8").splitlines()
]

required_input_fields = {"scenario", "variant", "step", "prompt", "ground_truth"}
if not QUERY_RECORDS:
    raise ValueError("The source query file contains no records.")

for row_number, query_record in enumerate(QUERY_RECORDS, start=1):
    if not isinstance(query_record, dict):
        raise ValueError(f"Query {row_number} is not an object.")
    if not required_input_fields.issubset(query_record):
        raise ValueError(f"Query {row_number} is missing required fields.")
    if not isinstance(query_record["prompt"], str):
        raise ValueError(f"Query {row_number} has a non-text prompt.")

# The raw download is no longer needed once the records have been parsed.
del query_download
print(f"Loaded {len(QUERY_RECORDS)} original queries.")
print(f"CAM-LDS revision: {CAM_LDS_REVISION}")

Loaded 990 original queries.
CAM-LDS revision: 44028d8bd40a4a1d8bbbc6ee33261d47cb433827


## 6. Construct the prompts using only the authors' cleanup

**Purpose:** Prepare the text sent to the models while preserving the original query records for CSV output.

The eight regular-expression substitutions below match the authors' inference script, in the same order. They remove the specified `key` fields and MITRE annotation fields from the prompt. Some of those `key` fields may not be MITRE labels; the notebook still follows the authors' substitutions exactly.

No extra cleaning follows: remaining technique IDs are only counted, whitespace is not separately normalised, and embedded alert JSON is not repaired. Each query will be sent as one user message, with no additional system prompt. The model's own GGUF chat template supplies its message formatting.

**CSV versus inference:** The CSV stores the unchanged original `prompt`. Inference receives its cleaned copy. Ground truth is never sent as an extra message or changed by this function.

Before delivery, all 990 source queries were checked against the authors' substitutions: 897 prompts changed, every cleaned prompt matched, and all other fields stayed unchanged.

In [9]:
CLEANUP_PATTERNS = [
    r'\s*key="[^"]*"',
    r'\s*key=\\"[^"]*\\"',
    r'\s*"key"\s*:\s*"[^"]*"',
    r',?\s*"mitre_tactic_id"\s*:\s*\[[^\]]*\]',
    r',?\s*"mitre_tactic_name"\s*:\s*\[[^\]]*\]',
    r',?\s*"mitre_technique_id"\s*:\s*\[[^\]]*\]',
    r',?\s*"mitre_technique_name"\s*:\s*\[[^\]]*\]',
    r',?\s*"mitre"\s*:\s*\{[^{}]*\}',
]

# Compile once; applying these rules produces the same text as re.sub.
CLEANUP_RULES = [re.compile(pattern) for pattern in CLEANUP_PATTERNS]


def construct_query(input_record):
    """Clean a copied prompt without modifying any original input fields."""
    prepared_record = copy.deepcopy(input_record)
    for cleanup_rule in CLEANUP_RULES:
        prepared_record["prompt"] = cleanup_rule.sub("", prepared_record["prompt"])
    return prepared_record


# Reuse prepared prompts across every model instead of cleaning repeatedly.
PREPARED_RECORDS = [construct_query(record) for record in QUERY_RECORDS]

changed_prompt_count = sum(
    original["prompt"] != prepared["prompt"]
    for original, prepared in zip(QUERY_RECORDS, PREPARED_RECORDS)
)
remaining_identifier_count = sum(
    bool(re.search(r"T\d{4}", record["prompt"]))
    for record in PREPARED_RECORDS
)

print(f"Prepared {len(PREPARED_RECORDS)} prompts.")
print(f"Prompts changed by the authors' cleanup: {changed_prompt_count}")
print(f"Prompts still containing technique IDs: {remaining_identifier_count}")

Prepared 990 prompts.
Prompts changed by the authors' cleanup: 897
Prompts still containing technique IDs: 0


## 7. Define the CAM-LDS response structure

**Purpose:** Require exactly the three response fields that will populate the final CSV columns.

`techniques` is a list of strings. `confidence` and `explanation` are strings. Additional fields are not allowed. This is the same schema as the authors' script; it does not add technique-ID filtering, confidence categories, or a fixed technique count.

The JSON grammar is built once and reused. After generation, the response is parsed and validated. Invalid output is not repaired. Instead, its query and source ID are omitted from the CSV and the run continues.

In [10]:
RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "techniques": {"type": "array", "items": {"type": "string"}},
        "confidence": {"type": "string"},
        "explanation": {"type": "string"},
    },
    "required": ["techniques", "confidence", "explanation"],
    "additionalProperties": False,
}

# Avoid rebuilding the JSON-schema grammar for each query.
RESPONSE_GRAMMAR = LlamaGrammar.from_json_schema(
    json.dumps(RESPONSE_SCHEMA), verbose=False,
)


def validate_response(response):
    """Check exactly the fields and types required by the CAM-LDS schema."""
    expected_fields = {"techniques", "confidence", "explanation"}
    if not isinstance(response, dict) or set(response) != expected_fields:
        raise ValueError("Unexpected response fields.")

    if not isinstance(response["techniques"], list) or not all(
        isinstance(value, str) for value in response["techniques"]
    ):
        raise ValueError("techniques must be a list of strings.")

    if not all(isinstance(response[key], str) for key in ["confidence", "explanation"]):
        raise ValueError("confidence and explanation must be strings.")

## 8. Define model downloading, loading, and query functions

**Purpose:** Download only the next unfinished model and keep it loaded for all of its queries.

`download_model` reuses the local Hugging Face cache and pins the revision used. It retains the dictionary in memory as `MODEL_REVISIONS`; inspect and copy it into Section 3 before restarting if you need those exact pins.

`load_model` requests all layers on the GPU, attention operations on the GPU, flash attention, and prompt-token batching. Engine verbosity is disabled so progress bars stay readable. To inspect actual layer placement while troubleshooting, temporarily set `verbose=True` in `load_model`; GPU support alone does not prove every layer was offloaded. A missing chat template or GPU support stops the run.

`run_inference` sends one user message with temperature 0 and the response grammar. `query` keeps the simple function interface from the original notebook. The same model object processes the entire query list; it is closed before the next model loads.

The default batch settings are a performance starting point, not a measured guarantee of maximum T4 throughput. If loading fails through insufficient GPU memory, try a microbatch of 256 and batch of 1024. Disable flash attention only if the backend reports it unsupported. These adjustments change execution settings, not the query text.

In [11]:
def download_model(model_key):
    """Download one pinned model, or reuse its existing local cache entry."""
    model_settings = MODELS[model_key]
    model_revision = MODEL_REVISIONS.get(model_key)

    if model_revision is None:
        model_revision = HfApi().model_info(model_settings["repo"]).sha
        MODEL_REVISIONS[model_key] = model_revision

    # MODEL_REVISIONS retains these pins in memory; inspect it only when needed.

    return Path(hf_hub_download(
        repo_id=model_settings["repo"],
        filename=model_settings["file"],
        revision=model_revision,
        cache_dir=str(MODEL_DIR),
    ))


def load_model(model_key):
    """Load one model using the GPU and selected performance settings."""
    if not llama_cpp.llama_supports_gpu_offload():
        raise RuntimeError(
            "The inference engine does not support GPU offloading."
        )

    model_settings = MODELS[model_key]
    context_size = CONTEXT_OVERRIDES.get(
        model_key, model_settings["ctx"]
    )
    model_path = download_model(model_key)

    model = Llama(
        model_path=str(model_path),
        n_ctx=context_size,

        # Request all model layers and attention operations on the GPU.
        n_gpu_layers=-1,
        main_gpu=0,
        offload_kqv=True,
        flash_attn=GPU_FLASH_ATTENTION,

        # Process long prompts using the sizes selected in Section 3.1.
        n_batch=min(GPU_BATCH_SIZE, context_size),
        n_ubatch=min(GPU_MICROBATCH_SIZE, context_size),

        # Limit CPU threads instead of automatically using every CPU.
        n_threads=CPU_THREADS,
        n_threads_batch=CPU_BATCH_THREADS,

        # Classification only needs next-token logits.
        logits_all=False,
        embedding=False,
        use_mmap=True,
        seed=SEED,
        verbose=False,
    )

    if not model.metadata.get("tokenizer.chat_template"):
        model.close()
        raise ValueError(
            f"{model_key}: the GGUF has no chat template."
        )

    return model


def run_inference(model, user_prompt):
    """Send a full prompt and return the complete local API result."""
    return model.create_chat_completion(
        messages=[{"role": "user", "content": user_prompt}],
        temperature=TEMPERATURE,
        max_tokens=MAX_OUTPUT_TOKENS,
        seed=SEED,
        grammar=RESPONSE_GRAMMAR,
    )


def query(model, prepared_record):
    """Retain the original notebook's function-based query interface."""
    return run_inference(model, prepared_record["prompt"])

In [19]:
# Register the same base model with a different quantisation.
# A separate key gives it a separate CSV, preserving the IQ1_M results.
MODELS["qwen1b_q4"] = {
    **MODELS["qwen1b"],
    "name": "Qwen2.5 1.5B Instruct Q4_K_M",
    "file": "Qwen2.5-1.5B-Instruct-Q4_K_M.gguf",
}

# Use the same context as the existing IQ1_M experiment.
CONTEXT_OVERRIDES["qwen1b_q4"] = CONTEXT_OVERRIDES.get(
    "qwen1b", MODELS["qwen1b"]["ctx"]
)

# Keep the same repository revision when it is already pinned.
if "qwen1b" in MODEL_REVISIONS:
    MODEL_REVISIONS["qwen1b_q4"] = MODEL_REVISIONS["qwen1b"]

# Test precisely the three prompts that produced repetitive failures.
compare_prompt_batches(
    model_key="qwen1b_q4",
    prompt_ids=(2, 89, 172),
)

Qwen2.5-1.5B-Instruct-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B /  986MB            

Qwen2.5-1.5B-Instruct-Q4_K_M.gguf: downloading bytes:           |  0.00B            

[{'configuration': 'previous',
  'batch': 2048,
  'microbatch': 512,
  'successful': 3,
  'failed': 0,
  'seconds': 9.67,
  'first error': ''},
 {'configuration': 'selected',
  'batch': 4096,
  'microbatch': 1024,
  'successful': 3,
  'failed': 0,
  'seconds': 10.44,
  'first error': ''}]

## 8.1 Compare prompt-processing performance
Run this check once when tuning a GPU. It compares the previous and selected batch sizes on identical prompts, using the existing context, generation settings, cleanup, and response validation. It displays elapsed time and failures without creating files. Skip this cell during routine resumes.
This measures inference on cold prompts; it excludes model-loading time and CSV-writing time. Three prompts provide an initial comparison, not a full-dataset benchmark.

In [12]:
from time import perf_counter


def compare_prompt_batches(
    model_key="qwen7b", #test with largest model
    prompt_ids=(1, 331, 661),
):
    """Compare inference settings without reading or writing CSVs."""
    global GPU_BATCH_SIZE, GPU_MICROBATCH_SIZE

    if not prompt_ids or any(
        not isinstance(prompt_id, int)
        or not 1 <= prompt_id <= len(PREPARED_RECORDS)
        for prompt_id in prompt_ids
    ):
        raise ValueError(
            "Choose valid one-based prompt IDs from the dataset."
        )

    selected_sizes = (GPU_BATCH_SIZE, GPU_MICROBATCH_SIZE)
    configurations = [
        ("previous", 2048, 512),
        ("selected", *selected_sizes),
    ]
    results = []

    try:
        for label, batch_size, microbatch_size in configurations:
            GPU_BATCH_SIZE = batch_size
            GPU_MICROBATCH_SIZE = microbatch_size
            model = None

            try:
                model = load_model(model_key)

                # Warm up each configuration before timing cold prompts.
                try:
                    query(
                        model,
                        PREPARED_RECORDS[prompt_ids[0] - 1],
                    )
                except (ValueError, KeyError, TypeError, IndexError):
                    pass

                successful_responses = 0
                first_error = ""
                start_time = perf_counter()

                for prompt_id in prompt_ids:
                    # Prevent reuse of the warm-up prompt prefix.
                    model.reset()

                    try:
                        result = query(
                            model,
                            PREPARED_RECORDS[prompt_id - 1],
                        )
                        choice = result["choices"][0]

                        if choice.get("finish_reason") == "length":
                            raise ValueError(
                                "Output reached its token/context limit."
                            )

                        response = json.loads(
                            choice["message"]["content"]
                        )
                        validate_response(response)
                        successful_responses += 1

                    except (
                        ValueError, KeyError, TypeError, IndexError
                    ) as error:
                        # Show only the first error in the final summary.
                        if not first_error:
                            first_error = f"Prompt {prompt_id}: {error}"

                results.append({
                    "configuration": label,
                    "batch": batch_size,
                    "microbatch": microbatch_size,
                    "successful": successful_responses,
                    "failed": len(prompt_ids) - successful_responses,
                    "seconds": round(
                        perf_counter() - start_time, 2
                    ),
                    "first error": first_error,
                })

            finally:
                if model is not None:
                    model.close()
                    gc.collect()

    finally:
        # Restore the selected settings, including after an interruption.
        GPU_BATCH_SIZE, GPU_MICROBATCH_SIZE = selected_sizes

    return results


compare_prompt_batches()

Qwen2.5-7B-Instruct-IQ1_M.gguf: reconstructing file:   0%|          |  0.00B / 2.04GB            

Qwen2.5-7B-Instruct-IQ1_M.gguf: downloading bytes:           |  0.00B            

[{'configuration': 'previous',
  'batch': 2048,
  'microbatch': 512,
  'successful': 3,
  'failed': 0,
  'seconds': 4.96,
  'first error': ''},
 {'configuration': 'selected',
  'batch': 2048,
  'microbatch': 512,
  'successful': 3,
  'failed': 0,
  'seconds': 4.94,
  'first error': ''}]

## 9. Define ID-based CSV writing and recovery

**Purpose:** Save valid responses with an unchanging link to their original source queries.

| CSV column | Source |
| --- | --- |
| `prompt id` | Original query position, starting at 1 |
| `scenario` | Original `scenario` |
| `variant` | Original `variant` |
| `step` | Original `step` |
| `prompt` | Unchanged original `prompt` |
| `ground truth` | Original `ground_truth` |
| `techniques` | Model response |
| `confidence` | Model response |
| `explanation` | Model response |

The main run cell automatically appends each successful response to this same CSV and flushes the write. Only successful responses appear in the CSV. If IDs 2 and 5 fail, those IDs are absent; later successes keep their original IDs. An empty predicted technique list is saved as `[]` and is still a valid response. Lists retain their values and order as JSON text inside CSV cells.

Resume validates strictly increasing IDs, maps each saved input to `QUERY_RECORDS[prompt_id - 1]`, and checks its response. It resumes at the next source position after the highest saved ID, not after the number of CSV records. For saved IDs `1, 3, 4`, the next prompt is 5.

An incomplete trailing CSV record is removed in place, without creating a backup. Earlier corruption, duplicate/out-of-range IDs, or mismatched inputs stop resume rather than silently changing saved results. Flushing/synchronising local writes cannot guarantee immediate remote Drive synchronisation.

**Progress limit:** A CSV containing only successful responses cannot remember failures after its highest saved ID. Such trailing failures may be attempted again following a restart. Ordinary gaps before that ID are treated as previously skipped inputs.

In [15]:
def csv_cell(value):
    """Represent a list or dictionary as JSON text inside one CSV cell."""
    return json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict)) else str(value)


def get_input_csv_values(record):
    """Return the five original input values, placed after prompt id in the CSV."""
    return [csv_cell(record[key]) for key in ['scenario', 'variant', 'step', 'prompt', 'ground_truth']]


def get_output_csv_path(model_key):
    """Identify a model and run through the filename in the shared folder."""
    return OUTPUT_FOLDER / f'llm_responses_{model_key}_camlds_{RUN_ID}.csv'


def append_csv_rows(csv_path, rows):
    """Append one batch and flush it without rewriting saved results."""
    if not rows:
        return
    # Format a whole checkpoint batch in memory before appending it.
    buffer = io.StringIO(newline='')
    csv.writer(buffer).writerows(rows)
    with csv_path.open('a', encoding='utf-8', newline='') as output:
        output.write(buffer.getvalue())
        output.flush()
        os.fsync(output.fileno())


def ensure_csv_exists(csv_path):
    """Create a CSV header or repair an interrupted initial header write."""
    buffer = io.StringIO(newline='')
    csv.writer(buffer).writerow(CSV_COLUMNS)
    # The UTF-8 marker helps spreadsheet applications open Unicode text.
    header = ('\ufeff' + buffer.getvalue()).encode('utf-8')
    if not csv_path.exists():
        with csv_path.open('xb') as output:
            output.write(header)
            output.flush()
            os.fsync(output.fileno())
    elif csv_path.stat().st_size < len(header):
        # Recover a header whose first write was interrupted.
        if not header.startswith(csv_path.read_bytes()):
            raise ValueError('The existing CSV has an incompatible header.')
        with csv_path.open('wb') as output:
            output.write(header)
            output.flush()
            os.fsync(output.fileno())



def read_csv_checkpoint(csv_path):
    """Validate sparse prompt IDs and return the highest ID and response count."""
    highest_prompt_id = 0
    saved_response_count = 0
    damaged_tail = False

    # Track byte positions to recover incomplete Unicode/multiline records.
    with csv_path.open("rb") as source:
        file_size = source.seek(0, 2)
        source.seek(max(0, file_size - 1))
        ends_with_newline = source.read(1) == b"\n"
        source.seek(0)

        def decoded_lines():
            first_line = True
            while line := source.readline():
                yield line.decode("utf-8-sig" if first_line else "utf-8")
                first_line = False

        reader = csv.reader(decoded_lines(), strict=True)
        if next(reader, None) != CSV_COLUMNS:
            raise ValueError(
                "Expected the nine-column CSV with prompt id first. "
                "Start a fresh run for an older CSV format; existing files are not imported."
            )

        last_valid_byte = source.tell()
        while True:
            try:
                row = next(reader)
            except StopIteration:
                break
            except (csv.Error, UnicodeDecodeError):
                if source.tell() != file_size:
                    raise ValueError("CSV damage occurs before the final row.")
                damaged_tail = True
                break

            if len(row) != len(CSV_COLUMNS) or (
                source.tell() == file_size and not ends_with_newline
            ):
                if source.tell() != file_size:
                    raise ValueError("An earlier CSV row is incomplete.")
                damaged_tail = True
                break

            # IDs are original one-based source positions, not CSV row numbers.
            if not row[0].isascii() or not row[0].isdigit():
                raise ValueError("CSV prompt IDs must be positive integers.")
            prompt_id = int(row[0])
            if not highest_prompt_id < prompt_id <= len(QUERY_RECORDS):
                raise ValueError("CSV prompt IDs must be in range and strictly increasing.")
            if row[1:6] != get_input_csv_values(QUERY_RECORDS[prompt_id - 1]):
                raise ValueError("A saved prompt ID does not match its original query.")

            try:
                validate_response({
                    "techniques": json.loads(row[6]),
                    "confidence": row[7],
                    "explanation": row[8],
                })
            except (ValueError, TypeError):
                if source.tell() != file_size:
                    raise ValueError("An earlier CSV response is invalid.")
                damaged_tail = True
                break

            highest_prompt_id = prompt_id
            saved_response_count += 1
            last_valid_byte = source.tell()

    if damaged_tail:
        # Remove only trailing partial data, without creating a backup file.
        with csv_path.open("r+b") as output:
            output.truncate(last_valid_byte)
            output.flush()
            os.fsync(output.fileno())

    return highest_prompt_id, saved_response_count


## 10. Define the unattended model runner

**Purpose:** Keep processing after ordinary query errors, while saving only valid responses with their original source IDs.

Each model has one bar showing its name, model position, percentage attempted, query count, elapsed time, ETA, CSV rows saved, failed attempts including gaps before the resume position, and status. With the default checkpoint interval of 1, each successful response is appended and flushed immediately. A normal cell interruption also attempts to save pending responses. Abrupt runtime loss during inference or writing may require that query to run again.

| Event | Behaviour |
| --- | --- |
| Valid model response | Save the three response values; continue |
| Context overflow or output limit | Skip its ID and CSV row; continue |
| Invalid JSON or response schema | Skip its ID and CSV row; continue |
| Caught model loading/engine failure | Save earlier pending rows, close the model, move to the next model |
| CSV write failure or mismatched checkpoint | Stop the experiment rather than continue without dependable saving |
| User stops the cell | Attempt to save pending rows, then stop |

The runner makes one inference attempt per query, instead of the authors' five-attempt hosted API retry loop. This is an execution-policy difference; query cleaning and the response schema are unchanged. A failed attempt creates no CSV row. The next successful response retains its original source ID.

The output-token cap bounds ordinary generation, but there is no hard wall-clock timeout for a hung native CUDA call. A native crash, runtime termination, or unrecoverable GPU state can still interrupt unattended execution.

**Progress display:** `status` moves from `loading` to `running`, then `done`. `already_saved` means the highest saved ID reaches the final source query. `model_failed` means the engine/loading problem stopped that model; the driver continues to the next. `interrupted` means a normal user stop. A CSV saving problem stops the cell and shows an exception traceback.

`failed` counts current failures plus gaps in the known source prefix on resume. Trailing failures after the last saved ID cannot be recovered from the success-only CSV. Percent complete counts attempted inputs, including failed ones. These values are display-only and never become CSV columns or extra files. There are no per-query error printouts or end-of-model timing printouts.

In [16]:
class ModelExecutionError(RuntimeError):
    """A loading or engine failure that prevents this model from continuing."""


def run_model(model_key, model_number=None):
    """Run one model with a percentage bar and no routine print messages."""
    csv_path = get_output_csv_path(model_key)
    ensure_csv_exists(csv_path)
    highest_saved_id, saved_response_count = read_csv_checkpoint(csv_path)
    # Prompt ID N corresponds to source index N-1; the next index is N.
    starting_row_index = highest_saved_id
    if model_number is None:
        model_number = SELECTED_MODELS.index(model_key) + 1

    model = None
    pending_rows = []
    saved_row_count = saved_response_count
    attempts_since_checkpoint = 0
    # Gaps before the resume position identify earlier skipped inputs.
    failed_query_count = starting_row_index - saved_response_count
    checkpoint_write_failed = False
    current_status = "loading"

    # Create the model bar before loading so download/loading time is visible.
    # The percentage counts all attempted queries, including failed attempts.
    with tqdm(
        total=len(QUERY_RECORDS),
        initial=starting_row_index,
        desc=f"{model_key} [{model_number}/{len(SELECTED_MODELS)}]",
        unit="query",
        mininterval=2,
        leave=True,
        bar_format=(
            "{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} "
            "[{elapsed}<{remaining}, {rate_fmt}{postfix}]"
        ),
    ) as progress:

        def update_status():
            """Keep useful run information inside this one progress bar."""
            progress.set_postfix(
                status=current_status,
                saved=saved_row_count,
                failed=failed_query_count,
                refresh=False,
            )

        def checkpoint():
            """Clear pending rows only after the CSV append succeeds."""
            nonlocal saved_row_count, checkpoint_write_failed
            if not pending_rows:
                return
            try:
                append_csv_rows(csv_path, pending_rows)
            except BaseException:
                # Resume must inspect any partly written batch before retrying.
                checkpoint_write_failed = True
                raise
            saved_row_count += len(pending_rows)
            pending_rows.clear()

        update_status()
        progress.refresh()

        try:
            # Skip loading when the source-position checkpoint reaches the end.
            if starting_row_index == len(QUERY_RECORDS):
                current_status = "already_saved"
                return

            try:
                model = load_model(model_key)
            except Exception as error:
                raise ModelExecutionError(
                    f"{model_key}: could not download/load the model: {error}"
                ) from error

            current_status = "running"
            update_status()
            progress.refresh()

            for row_index in range(starting_row_index, len(QUERY_RECORDS)):
                response = None
                try:
                    result = query(model, PREPARED_RECORDS[row_index])
                    choice = result["choices"][0]
                    if choice.get("finish_reason") == "length":
                        raise ValueError("Output reached its token or context limit.")

                    # Parse the original model text without repairing its JSON.
                    parsed_response = json.loads(choice["message"]["content"])
                    validate_response(parsed_response)
                    response = parsed_response

                except Exception as error:
                    # Engine failures can invalidate CUDA state. Hand off to
                    # the next model rather than keep using the broken instance.
                    if isinstance(error, (RuntimeError, MemoryError, OSError)):
                        raise ModelExecutionError(
                            f"{model_key}: engine failed at query {row_index + 1}: {error}"
                        ) from error

                    # Skip this query: do not write a row or renumber later IDs.
                    # Increment the bar's failure counter instead of printing.
                    failed_query_count += 1

                if response is not None:
                    # This ID always refers to the original llm_queries.json row.
                    # Failed queries leave gaps, such as 1, 3, 4 when query 2 fails.
                    pending_rows.append(
                        [row_index + 1]
                        + get_input_csv_values(QUERY_RECORDS[row_index])
                        + [
                            json.dumps(response["techniques"], ensure_ascii=False),
                            response["confidence"],
                            response["explanation"],
                        ]
                    )

                # Progress counts every attempt; the CSV counts valid responses.
                attempts_since_checkpoint += 1
                progress.update(1)
                if attempts_since_checkpoint >= CHECKPOINT_EVERY:
                    checkpoint()
                    attempts_since_checkpoint = 0
                update_status()

            current_status = "done"

        except ModelExecutionError:
            current_status = "model_failed"
            raise
        except KeyboardInterrupt:
            current_status = "interrupted"
            raise
        except BaseException:
            current_status = "save_failed" if checkpoint_write_failed else "stopped"
            raise
        finally:
            try:
                # Save the final partial batch on completion or a normal stop.
                if not checkpoint_write_failed:
                    checkpoint()
            except BaseException:
                current_status = "save_failed"
                raise
            finally:
                try:
                    if model is not None:
                        model.close()
                        gc.collect()
                finally:
                    # Leave the final saved count and run state on the bar.
                    update_status()
                    progress.refresh()

## 11. Run all selected models in the background

**Purpose:** Run the selected models sequentially without requiring you to handle each bad query.

Start this long-running cell after setup. You can work elsewhere while the runtime remains active. Ordinary query errors skip their CSV row and ID, and the dataset continues. A caught model-level failure moves to the next model. A final 100% bar means inputs were attempted; it does not mean every query succeeded. The bar retains its failure count.

Rerunning this cell uses the current run ID. Resume skips IDs through the highest saved ID; trailing failures may be retried after restart. A model whose known source-position checkpoint reaches the end is skipped before downloading or loading its weights. Only one execution should write to a given CSV; stop the existing run before starting another.

**Colab lifetime:** Error handling does not keep Colab alive or restart a deleted runtime. Colab controls runtime lifetimes and compute availability. Its official FAQ documents continuous execution for up to 24 hours on Pro+ with sufficient compute units. Background execution with the browser closed depends on the benefits of your plan; do not assume it for an ordinary/free session.

The bar shows ETA and attempt rate. Routine model/query printouts are removed, and engine verbosity is off. Hugging Face may show its own weight-download progress. Caught model failures are marked on their bars; if needed, inspect the in-memory `model_errors` dictionary after the run. No benchmark or error files are created.

In [17]:
# Each model has its own percentage bar. Failed models remain marked on screen.
# Ordinary query failures continue inside run_model, without extra print lines.
# CSV errors still stop execution and show their exception traceback.
stopped_models = []
model_errors = {}

for model_number, model_key in enumerate(SELECTED_MODELS, start=1):
    try:
        run_model(model_key, model_number=model_number)
    except ModelExecutionError as error:
        stopped_models.append(model_key)
        model_errors[model_key] = str(error)
        # The model's bar already shows model_failed. Start the next model.

qwen1b [1/5]:  17%|#7        | 171/990 [00:00<?, ?query/s]

Qwen2.5-1.5B-Instruct-IQ1_M.gguf: reconstructing file:   0%|          |  0.00B /  464MB            

Qwen2.5-1.5B-Instruct-IQ1_M.gguf: downloading bytes:           |  0.00B            

KeyboardInterrupt: 

In [18]:
# These first 173 prompts were already attempted.
last_attempted_id = 173

# Read only qwen1b's existing output CSV.
csv_path = get_output_csv_path("qwen1b")
with csv_path.open(encoding="utf-8-sig", newline="") as source:
    saved_ids = {
        int(row["prompt id"])
        for row in csv.DictReader(source)
    }

failed_ids = [
    prompt_id
    for prompt_id in range(1, last_attempted_id + 1)
    if prompt_id not in saved_ids
]

# Sample early, middle, and recent failures.
test_ids = sorted({
    failed_ids[0],
    failed_ids[len(failed_ids) // 2],
    failed_ids[-1],
})

diagnostic_results = []
model = load_model("qwen1b")

try:
    for prompt_id in test_ids:
        try:
            result = query(model, PREPARED_RECORDS[prompt_id - 1])
            choice = result["choices"][0]
            response_text = choice["message"]["content"]

            # Check validity even when generation reached its token limit.
            validation_error = ""
            try:
                validate_response(json.loads(response_text))
            except (ValueError, TypeError) as error:
                validation_error = str(error)

            diagnostic_results.append({
                "prompt id": prompt_id,
                "finish reason": choice.get("finish_reason"),
                "tokens": result.get("usage"),
                "validation error": validation_error,
                "response start": response_text[:400],
                "response end": response_text[-400:],
            })

        except Exception as error:
            diagnostic_results.append({
                "prompt id": prompt_id,
                "error": f"{type(error).__name__}: {error}",
            })

            # Avoid continuing with a potentially broken engine.
            if isinstance(error, (RuntimeError, MemoryError, OSError)):
                break

finally:
    model.close()
    gc.collect()

diagnostic_results

[{'prompt id': 2,
  'finish reason': 'length',
  'tokens': {'prompt_tokens': 1026,
   'completion_tokens': 2048,
   'total_tokens': 3074},
  'validation error': 'Unterminated string starting at: line 1 column 749 (char 748)',
  'response start': '{ "techniques": ["ATT", "ATT", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL", "NORMAL",',
  'response end': 'we are, it is not of the type we are, it is not of the type we are, it is not of the type we are, it is not of the type we are, it is not of the type we are, it is not of the type we are, it is not of the type we are, it is not of the type we are, it is not of the type we are, it is not of the type we are, it is not of the type w

## 12. Stop, resume, or start a new experiment

### Stop and resume in the same runtime

Stop the running cell and let it finish stopping. The runner attempts to flush pending successful responses; the interrupted query is not saved. Rerun Section 11 with the same run ID. A browser disconnection does not always terminate execution, so check whether the existing cell is still running before starting another.

### Resume after a runtime restart

Set `RESUME_RUN_ID` to the timestamp in your current nine-column output CSV filename. This reads only the model's own output CSV to find its resume position; no other CSV import is needed. Restore the dataset revision, model revision pins, and generation/context settings. Run setup, then Section 11. Saved IDs remain unchanged. The highest saved ID determines the next source position; failed IDs before it stay skipped. Failed tail queries after it may be retried, because no separate failure/checkpoint file exists.

### Retry skipped queries under new settings

Missing IDs identify skipped inputs in the prefix already attempted; IDs not reached yet are also absent. Therefore an absent ID alone does not prove the query failed while a run is still incomplete. A 100% progress bar means all source queries were attempted during that execution, even if the CSV has fewer than 990 rows.

Start a separate experiment to retry under changed context/output settings without mixing configurations in one CSV. The notebook does not shorten queries or repair invalid model output.
Older eight-column CSVs require a fresh run ID for this notebook. They are not imported or modified.


## Verification and source notes

Before delivery, the notebook code was checked for syntax, cell order, and helper dependencies. The cleanup was compared with the pinned authors' script across all 990 queries, including equality of every other input field. The response schema was checked against the authors' schema.

CSV tests covered Unicode, quoted multiline text, interrupted row writes, partial headers, changed-input rejection, normal interruption, and resume without duplicate saved rows. Mocked inference was used for recovery tests, including failed-query continuation, sparse-ID resume, engine-failure handoff, and CSV-write-error propagation. The notebook was not run end-to-end on a Colab T4 here, so model throughput and full dataset completion remain runtime checks.

Primary sources:

- [CAM-LDS authors' inference script at the pinned revision](https://github.com/ait-aecid/attack-manifestations-interpretation/blob/44028d8bd40a4a1d8bbbc6ee33261d47cb433827/get_llm_responses.py)
- [Original query file at the same revision](https://github.com/ait-aecid/attack-manifestations-interpretation/blob/44028d8bd40a4a1d8bbbc6ee33261d47cb433827/llm_queries.json)
- [Official CUDA 12.4 wheel index](https://abetlen.github.io/llama-cpp-python/whl/cu124/llama-cpp-python/)
- [llama-cpp-python API reference](https://llama-cpp-python.readthedocs.io/en/latest/api-reference/)
- [Google Colab official FAQ: runtime limits and background execution](https://research.google.com/colaboratory/faq.html)
Progress-display tests also check percentage-bar labels, incremental counts, loading/completion/error states, and the absence of routine prints in the execution functions.
Prompt-ID and saving tests check immediate writes after successful queries, nine-column output, original row mappings, gaps after failures, resume from the highest saved ID, interrupted writes.